제공해주신 2시간 분량의 강의 교안을 바탕으로, 수강생들이 Google Colab 환경에서 순차적으로 실행하며 학습할 수 있는 Python 실습 코드를 작성했습니다. 실제 교육 환경에 바로 도입하실 수 있도록, 교안에 명시된 흐름과 핵심 실습(샘플링, 파라미터 제어, 스트리밍, 구조화 출력, 미니 프로젝트)을 모두 포함하여 셀(Cell) 단위로 구성했습니다.

---

### [Cell 1] 환경 설정 및 보안 (API 키 로드)

Colab의 보안 환경(`Secrets`)에서 API 키를 안전하게 불러오고 필수 라이브러리를 설치하는 단계입니다. 실습에 사용할 최신/고전 모델 변수도 함께 정의합니다.

In [2]:
# 필수 라이브러리 설치
!pip -q install -U openai pydantic pandas

from google.colab import userdata
import os
from openai import OpenAI

# Colab 왼쪽 툴바의 '열쇠' 아이콘(Secrets)에 OPENAI_API_KEY를 등록해야 합니다.
key = userdata.get("OPENAI_API_KEY")
if not key:
    raise RuntimeError("Colab Secrets에 OPENAI_API_KEY를 등록하세요.")

os.environ["OPENAI_API_KEY"] = key
client = OpenAI()

# 실습용 모델 설정 (시점 및 계정에 따라 수정 가능)
MODEL_CURRENT = "gpt-5.6-luna"   # 현재형 실습 (최신 모델 기준)
MODEL_SAMPLING = "gpt-4o-mini"   # 고전 샘플링 비교 실습

print("환경 설정 완료 및 API 클라이언트 초기화 성공!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 115.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.
환경 설정 완료 및 API 클라이언트 초기화 성공!


### [Cell 2] 첫 Chat Completion 요청

가장 기본적인 `messages` 배열(developer, user 역할)을 구성하고, 응답 객체의 핵심 내용(`content`, `finish_reason`, `usage`)을 관찰합니다.

In [4]:
completion = client.chat.completions.create(
    model=MODEL_CURRENT,
    messages=[
        {
            "role": "developer",
            "content": "당신은 생성형 AI 과정의 친절한 튜터입니다. 한국어로 답하세요." # [cite: 41]
        },
        {
            "role": "user",
            "content": "temperature를 비전공자도 이해하게 3문장으로 설명해 주세요." # [cite: 41]
        }
    ],
    max_completion_tokens=250, # [cite: 41]
)

print("[생성된 텍스트]")
print(completion.choices[0].message.content) # 사용자에게 보여 줄 텍스트 [cite: 46]
print("\n[메타데이터]")
print("finish_reason:", completion.choices[0].finish_reason) # 종료 이유 [cite: 47]
print("usage:", completion.usage) # 토큰 사용량 [cite: 48]

[생성된 텍스트]
Temperature는 AI가 답변을 만들 때 얼마나 “다양하고 자유롭게” 선택할지를 조절하는 값입니다.  
낮게 설정하면 답변이 일관되고 정확한 편이고, 높게 설정하면 새롭고 창의적이지만 엉뚱한 답이 나올 가능성도 커집니다.  
예를 들어 요리법처럼 정확성이 중요하면 낮게, 아이디어를 많이 내야 하면 높게 설정합니다.

[메타데이터]
finish_reason: stop
usage: CompletionUsage(completion_tokens=101, prompt_tokens=50, total_tokens=151, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0))


### [Cell 3] 멀티턴 대화와 상태 관리

이전 응답 객체를 `messages` 배열에 추가하여 대화의 맥락(상태)을 유지하는 방법을 실습합니다.

In [6]:
messages = [
    {"role": "developer", "content": "간결한 한국어 API 튜터로 답하세요."}, # [cite: 51]
    {"role": "user", "content": "temperature를 한 문장으로 정의해 주세요."}, # [cite: 51]
]

# 첫 번째 요청
first = client.chat.completions.create(
    model=MODEL_CURRENT,
    messages=messages,
    max_completion_tokens=120, # [cite: 51]
)
print("[첫 번째 답변]\n", first.choices[0].message.content)

# 대화 상태 업데이트 [cite: 50, 51]
messages.append(first.choices[0].message)
messages.append({
    "role": "user",
    "content": "방금 설명을 쇼핑몰 상품명 생성 예시로 바꿔 주세요." # [cite: 51]
})

# 두 번째 요청
second = client.chat.completions.create(
    model=MODEL_CURRENT,
    messages=messages,
    max_completion_tokens=180, # [cite: 51]
)
print("\n[두 번째 답변 (맥락 유지)]\n", second.choices[0].message.content) # [cite: 51]

[첫 번째 답변]
 Temperature는 시스템의 열에너지 수준을 나타내는 물리량입니다.

[두 번째 답변 (맥락 유지)]
 상품명 예시: **정밀 온도 측정 센서｜시스템의 열에너지 수준 감지**


### [Cell 4] 샘플링 파라미터 제어 (Temperature vs Top_p)

`ask`라는 재사용 가능한 헬퍼 함수를 정의하고, 변인을 하나씩만 통제하여 답변의 다양성을 비교합니다.

In [8]:
# 비교 실험을 위한 헬퍼 함수
def ask(model, prompt, max_tokens=120, **params):
    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "developer", "content": "한국어 카피라이터. 결과만 출력."}, #
            {"role": "user", "content": prompt},
        ],
        max_completion_tokens=max_tokens,
        n=1, # 실습 비용 안전장치
        **params,
    )
    return result.choices[0].message.content, result

prompt = "무설탕 탄산수 신제품 이름과 한 줄 카피를 3개 제안하세요." #

print("=== Temperature 실험 ===")
for temp in [0.2, 0.8, 1.2]:
    text, result = ask(MODEL_SAMPLING, prompt, temperature=temp) #
    print(f"\n--- temperature={temp} ---\n{text}")

print("\n=== Top_p 실험 (Temperature는 기본값) ===")
# 두 값을 동시에 바꾸면 원인을 분리하기 어려우므로 택일하여 조정합니다.
for p in [0.2, 0.7, 1.0]:
    text, result = ask(MODEL_SAMPLING, prompt, top_p=p)
    print(f"\n--- top_p={p} ---\n{text}")

=== Temperature 실험 ===

--- temperature=0.2 ---
1. **청량한 자유**  
   "설탕 없이도 상쾌함을 만끽하세요!"

2. **스파클링 퓨어**  
   "순수한 맛, 탄산의 즐거움!"

3. **제로 스파크**  
   "당신의 갈증을 깨우는 무설탕 탄산수!"

--- temperature=0.8 ---
1. **청량의 비밀**  
   "자연의 맛을 그대로 담은 무설탕 탄산수!"

2. **스파클링 퓨어**  
   "순수함이 터지는, 달콤함은 없다."

3. **제로 버블**  
   "설탕 없이도 풍성한 기쁨, 제로 버블!"

--- temperature=1.2 ---
1. 이름: "푸른흐름"  
   카피: "자연의 신선함을 담은 무설탕 탄산수, 푸른흐름과 함께하세요."

2. 이름: "청량한 순간"  
   카피: "저칼로리의 정수, 청량한 순간으로 나를 리프레시!"

3. 이름: "스파클링 퓨어"  
   카피: "설탕 없는 깨끗한 맛, 스파클링 퓨어로 매일을 특별하게

=== Top_p 실험 (Temperature는 기본값) ===

--- top_p=0.2 ---
1. **청량한 바람**  
   "자연의 청량함을 담은 무설탕 탄산수, 당신의 갈증을 시원하게 날려보세요."

2. **스파클링 퓨어**  
   "순수한 맛, 탄산의 기쁨! 무설탕으로 더 건강하게 즐기세요."

3. **프레시 블라스트**  
   "상쾌함이 터지는 순간, 무설탕 탄산수로 기분을 리프레시하세요!"

--- top_p=0.7 ---
1. **청량한 순간**  
   "무설탕으로 더 맑고, 더 상쾌한 기분을 느껴보세요!"

2. **스파클링 퓨어**  
   "자연의 청량함을 담은, 당신만을 위한 탄산수!"

3. **제로버블**  
   "설탕 없는 기쁨, 탄산의 상쾌함을 그대로!"

--- top_p=1.0 ---
1. 이름: 에어리프레쉬  
   카피: "상쾌함이 터지는 순간, 무설탕의 새로운 즐거움!"

2. 이름: 퓨어

### [Cell 5] 길이 제한 및 페널티 (Length, Frequency, Presence)

출력 상한(`max_completion_tokens`) 도달 시의 에러 관찰 및 두 가지 페널티의 차이를 비교합니다.

In [10]:
print("=== 길이 제한(length) 도달 확인 ===")
text_len, result_len = ask(
    MODEL_SAMPLING,
    "Chat Completions 비용 절감 원칙을 5개 설명하세요.", # [cite: 76]
    temperature=0.2,
    max_tokens=80, # [cite: 76]
)
# finish_reason이 'length'이면 상한 도달을 의미합니다. [cite: 77]
print("종료 이유:", result_len.choices[0].finish_reason) # [cite: 76]

print("\n=== 페널티(Penalty) 실험 ===")
base = {
    "temperature": 0.7,
    "frequency_penalty": 0.0,
    "presence_penalty": 0.0,
} # [cite: 80]

# baseline, frequency(반복 억제), presence(새 주제 유도) 비교 [cite: 79, 80]
for name, override in [
    ("baseline", {}),
    ("frequency", {"frequency_penalty": 0.6}),
    ("presence", {"presence_penalty": 0.6}),
]:
    params = {**base, **override}
    text, _ = ask(MODEL_SAMPLING, "AI 교육 과정 주제를 8개 제안하세요.", **params) # [cite: 80]
    print(f"\n--- {name} ---\n{text}")

=== 길이 제한(length) 도달 확인 ===
종료 이유: length

=== 페널티(Penalty) 실험 ===

--- baseline ---
1. 기초 인공지능 개념과 원리
2. 머신러닝 알고리즘의 이해와 적용
3. 딥러닝을 활용한 이미지 인식
4. 자연어 처리(NLP) 기초와 실습
5. 데이터 전처리 및 분석 기술
6. 인공지능 윤리와 사회적 영향
7. AI 기반 프로젝트 관리와 실행
8. 인공지능 툴과 플랫폼 활용법

--- frequency ---
1. 인공지능 기초: 개념과 역사
2. 머신러닝의 원리와 응용
3. 딥러닝을 활용한 이미지 인식
4. 자연어 처리(NLP)와 챗봇 개발
5. 데이터 분석과 AI: 실전 사례 연구
6. 윤리적 AI: 책임 있는 기술 사용
7. AI를 활용한 비즈니스 혁신 전략
8. 로봇 공학과 AI의 미래 전망

--- presence ---
1. 인공지능 기초: AI의 역사와 원리
2. 머신러닝 입문: 데이터 분석과 예측 모델링
3. 딥러닝 심화: 신경망 구조와 응용
4. 자연어 처리(NLP): 언어 이해와 생성 기술
5. 컴퓨터 비전: 이미지 인식 및 처리 기술
6. AI 윤리: 인공지능의 사회적 책임과 도전 과제
7. AI와 빅데이터: 데이터 활용 전략 및 사례
8. 로봇 공학: AI


### [Cell 6] Logprobs 관찰

토큰별 로그 확률을 반환받아 모델의 불확실성을 확인하는 디버깅 기능입니다.

In [13]:
result = client.chat.completions.create(
    model=MODEL_SAMPLING,
    messages=[
        {"role": "developer", "content": "긍정 또는 부정 한 단어만 출력하세요."}, # [cite: 83]
        {"role": "user", "content": "이 수업은 이해하기 쉽고 유익했다."}, # [cite: 83]
    ],
    temperature=0,
    max_completion_tokens=10,
    logprobs=True, # 로그 확률 반환 활성화 [cite: 59, 83]
    top_logprobs=3, # 각 위치의 상위 후보 3개 [cite: 59, 83]
)

choice = result.choices[0]
print("최종 출력:", choice.message.content) # [cite: 83]

print("\n[토큰별 Logprobs 분석]")
# 0에 가까울수록 높은 확률을 의미합니다. [cite: 85]
for item in choice.logprobs.content:
    print(f"토큰: '{item.token}' | 확률(logprob): {item.logprob}") # [cite: 83]
    print("상위 후보:", [(x.token, x.logprob) for x in item.top_logprobs]) # [cite: 83]

최종 출력: 긍정

[토큰별 Logprobs 분석]
토큰: '\xea\xb8' | 확률(logprob): -9.448370838072151e-05
상위 후보: [('\\xea\\xb8', -9.448370838072151e-05), (' \\xea\\xb8', -9.500094413757324), ('부', -12.500094413757324)]
토큰: '\x8d' | 확률(logprob): -3.128163257315464e-07
상위 후보: [('\\x8d', -3.128163257315464e-07), ('\\x81', -15.625), ('\\x8b', -17.875)]
토큰: '정' | 확률(logprob): -1.8624639324116288e-06
상위 후보: [('정', -1.8624639324116288e-06), ('적', -14.000001907348633), (' 정', -15.000001907348633)]


### [Cell 7] 스트리밍 출력 (SSE) 및 사용량 확인

점진적으로 텍스트를 생성하여 보여주는 스트리밍 방식과, 마지막 Chunk에서 토큰 사용량을 함께 확인하는 방법입니다.

In [15]:
stream = client.chat.completions.create(
    model=MODEL_CURRENT,
    messages=[{"role": "user", "content": "생성형 AI 수업 오프닝 멘트를 작성하세요."}], # [cite: 90]
    max_completion_tokens=180,
    stream=True, # 점진적 출력 [cite: 59, 90]
    stream_options={"include_usage": True}, # 종료 직전 usage chunk 반환 [cite: 90, 91]
)

print("스트리밍 출력 중...")
for chunk in stream:
    if chunk.choices:
        delta = chunk.choices[0].delta.content
        if delta:
            print(delta, end="", flush=True) # [cite: 90]

    # [cite_start]마지막 chunk에서 사용량 출력 [cite: 90, 91]
    if chunk.usage:
        print("\n\n[최종 토큰 사용량]")
        print("usage:", chunk.usage) # [cite: 90]

스트리밍 출력 중...
안녕하세요, 여러분. 오늘 생성형 AI 수업에 함께해 주셔서 감사합니다.

생성형 AI는 단순히 질문에 답하는 도구를 넘어, 글을 쓰고, 이미지를 만들고, 아이디어를 확장하며, 우리가 일하고 배우는 방식을 바꾸고 있습니다. 하지만 중요한 것은 AI를 사용하는 것 자체가 아니라, **어떤 질문을 던지고, 결과를 어떻게 판단하며, 나의 목적에 맞게 어떻게 활용하느냐**입니다.

오늘 수업에서는 생성형 AI의 기본 원리를 이해하고, 실제 사례와 실습을 통해 효과적으로 질문하는 방법, 원하는 결과를 얻기 위한 프롬프트 작성법

[최종 토큰 사용량]
usage: CompletionUsage(completion_tokens=180, prompt_tokens=21, total_tokens=201, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=27, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0))


### [Cell 8] Structured Outputs (구조화 출력)

Pydantic을 활용하여 애플리케이션 연동에 필수적인 JSON Schema 기반 구조화 데이터를 100% 보장하여 반환받는 실습입니다.

In [17]:
from typing import Literal
from pydantic import BaseModel, Field # [cite: 97]

# Pydantic을 이용한 스키마 정의 [cite: 97]
class SupportTicket(BaseModel):
    category: Literal["결제", "기술", "계정", "기타"] # [cite: 97]
    priority: Literal["low", "medium", "high"] # [cite: 97]
    summary: str = Field(description="한국어 한 문장 요약") # 설명이 필요한 필드 [cite: 97, 100]

# .parse() 메서드 사용 [cite: 97]
completion = client.chat.completions.parse(
    model=MODEL_CURRENT,
    messages=[
        {"role": "developer", "content": "고객 문의를 정확히 분류하세요."}, # [cite: 97]
        {"role": "user", "content": "결제는 됐는데 강의 영상이 열리지 않아요."}, # [cite: 97]
    ],
    response_format=SupportTicket, # 앱 계약 명시 [cite: 96, 97]
)

ticket = completion.choices[0].message.parsed # 파싱된 결과 객체 [cite: 97]
print("[구조화된 출력 결과]")
print(ticket.model_dump()) # [cite: 97]

[구조화된 출력 결과]
{'category': '기술', 'priority': 'high', 'summary': '결제는 완료되었지만 구매한 강의 영상이 열리지 않습니다.'}


### [Cell 9] 미니 프로젝트: 강의 Q&A 도우미

강의 목표 중 하나인 "독립적인 과업 수행"을 위한 마무리 미니 프로젝트 코드입니다.

In [19]:
from pydantic import BaseModel
from typing import Literal

# 1. 스키마 정의 [cite: 123]
class QnAResponse(BaseModel):
    level: Literal["beginner", "intermediate"] # [cite: 123]
    answer: str
    next_exercise: str

def get_qna_assistance(question, verbosity="medium"):
    # 2. 파라미터 제어와 시스템 프롬프트 (developer 메시지에 역할, 수준, 금지사항 명시) [cite: 122]
    try:
        response = client.chat.completions.parse(
            model=MODEL_CURRENT,
            messages=[
                {"role": "developer", "content": "당신은 Python 튜터입니다. 한국어로 대상 수준에 맞게 3문장 이내로 답변하고, 후속 실습을 제안하세요. 정답을 직접적으로 알려주지 마세요."}, # [cite: 121, 122]
                {"role": "user", "content": question}
            ],
            response_format=QnAResponse,
            max_completion_tokens=300
        )
        return response.choices[0]
    except Exception as e:
        print("오류 발생:", e)
        return None

# 3. 질문 테스트 [cite: 124]
test_questions = [
    "반복문 while과 for의 차이가 뭐죠?",
    "리스트 컴프리헨션에서 조건문은 어떻게 쓰나요?"
]

for q in test_questions:
    print(f"\nQ: {q}")
    result = get_qna_assistance(q)
    if result and result.message.parsed:
        parsed_data = result.message.parsed
        print(f"난이도: {parsed_data.level}")
        print(f"답변: {parsed_data.answer}")
        print(f"후속 실습: {parsed_data.next_exercise}")
        print(f"토큰 사용량: {result.message.content if hasattr(result, 'usage') else '확인 필요'}") # 메타데이터 확인 (평가 루브릭 반영) [cite: 127]


Q: 반복문 while과 for의 차이가 뭐죠?
난이도: beginner
답변: `for`는 리스트나 문자열처럼 정해진 대상의 항목을 하나씩 처리할 때 주로 쓰고, `while`은 조건이 참인 동안 반복할 때 사용합니다. 반복 횟수가 정해져 있거나 순회 대상이 있으면 `for`, 종료 조건을 직접 관리해야 하면 `while`이 알맞습니다.
후속 실습: 1부터 10까지 출력하는 코드를 `for`와 `while`로 각각 작성해 보세요.
토큰 사용량: 확인 필요

Q: 리스트 컴프리헨션에서 조건문은 어떻게 쓰나요?
난이도: beginner
답변: 리스트 컴프리헨션에서는 `for` 반복 뒤에 `if` 조건을 붙여 조건에 맞는 값만 골라낼 수 있습니다. 예를 들어 1부터 10까지의 수 중 짝수만 남기는 코드를 직접 만들어 보세요.
후속 실습: `numbers = [1, 2, 3, 4, 5]`에서 3보다 큰 수만 추출하는 리스트 컴프리헨션을 작성해 보세요.
토큰 사용량: 확인 필요
